# NLP Laboratory Assignment 04: Country Prediction using Word Embeddings

**Student:** Affan Khan Samid Khan  
**Roll No.:** 421135  
**PRN:** 22310588  
**Faculty:** Dr. Kapil N. Vhatkar

This notebook evaluates 20 capital-country analogies using the pre-trained GloVe Wiki-Gigaword 50-dimensional vectors through Gensim. On first run, the vector package (about 65 MB) must be downloaded; later runs can use the local cache.

**Important:** the authoring environment could not download the model weights, so no model predictions or scores have been fabricated. Execute the model cells in an internet-enabled Python environment to generate the real numeric results.

In [ ]:
from pathlib import Path
import os, subprocess
SOURCE_NAME = "421135_Affan_Khan_Samid_Khan_NLP_Assignment04.py"
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / SOURCE_NAME).exists():
    candidates = [PROJECT_DIR / "Assignment04", PROJECT_DIR.parent / "Assignment04"]
    found = next((p for p in candidates if (p / SOURCE_NAME).exists()), None)
    if found is not None:
        PROJECT_DIR = found
    else:
        repo_dir = Path("/content/NLP-assignments") if Path("/content").exists() else PROJECT_DIR / "NLP-assignments"
        if not repo_dir.exists():
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/affanSkhan/NLP-assignments.git", str(repo_dir)], check=True)
        PROJECT_DIR = repo_dir / "Assignment04"
if not (PROJECT_DIR / SOURCE_NAME).exists():
    raise FileNotFoundError("Place this notebook inside Assignment04 or clone the repository.")
os.chdir(PROJECT_DIR)
print("Project directory:", PROJECT_DIR.resolve())
print("Evaluation set:", (PROJECT_DIR / "analogy_test_cases.csv").resolve())


In [ ]:
%pip install -r requirements.txt


## Load the pre-trained model and evaluate all analogies

The model is loaded through Gensim. If an input or expected answer is out of vocabulary, the example is recorded as skipped rather than silently scored.

In [ ]:
import importlib.util, sys, json
import pandas as pd
from IPython.display import display
spec = importlib.util.spec_from_file_location("assignment04", PROJECT_DIR / SOURCE_NAME)
assignment04 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = assignment04
spec.loader.exec_module(assignment04)
wv = assignment04.load_model()
metrics, predictions, rankings = assignment04.run_evaluation(
    wv, PROJECT_DIR / "analogy_test_cases.csv", PROJECT_DIR / "outputs", topn=5
)
print(json.dumps(metrics, indent=2))
display(pd.DataFrame(predictions))


## Top-five candidates for the first five analogies

In [ ]:
ranked_df = pd.DataFrame(rankings)
display(ranked_df[ranked_df["example_id"].isin([1, 2, 3, 4, 5])])


## Required example: Athens : Greece :: Baghdad : ?

In [ ]:
example = assignment04.predict_country("Athens", "Greece", "Baghdad", wv, topn=5)
print(json.dumps(example, indent=2))


## Interpretation and limitations

- The analogy vector is approximate; country-capital relations need not form a perfect linear offset in embedding space.
- Candidate ranking is restricted to a transparent list of country-name tokens.
- Cosine similarity is a geometric score, not a probability.
- The evaluation set is small and manually curated; it is not a standard benchmark.
- Out-of-vocabulary inputs and target answers are reported as skipped.

Generated files are saved in `outputs/`: `predictions.csv`, `ranked_candidates.csv`, `skipped_oov_examples.csv`, `metrics.json`, `results.txt`, and (with Matplotlib) `cosine_scores.png`.